In [0]:
#Explicacao SEQUENCE E EXPLODE
# 1) sequence: ONE row holding a list of days
spark.sql("""
    SELECT sequence(to_date('2026-10-09'), to_date('2026-10-11'), interval 1 day) AS days
""").show(truncate=False)

# 2) explode: the list becomes ONE ROW PER DAY
spark.sql("""
    SELECT explode(sequence(to_date('2026-10-09'), to_date('2026-10-11'), interval 1 day)) AS day
""").show()

In [0]:
# Read the silver table created in the previous stage
silver = spark.table("workspace.silver.transactions")

# 1) Column names and types
silver.printSchema()

# 2) First rows, full width
silver.show(5, truncate=False)

# 3) How many distinct values each column has
# Few distinct values -> dimension candidate. Unique per row -> identifier/measure.
for c in silver.columns:
    print(c, "->", silver.select(c).distinct().count())

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.silver.transactions")

# Earliest and latest transaction date: the calendar must cover this whole range
bounds = silver.agg(
    F.min("transaction_date").alias("min_d"),
    F.max("transaction_date").alias("max_d"),
).first()

print(bounds["min_d"], bounds["max_d"])

In [0]:
df_dates = (
    spark.range(1)  # a dummy 1-row table, only to host the expression below
    .select(
        # sequence() builds the list of days; explode() turns the list into rows
        F.explode(
            F.sequence(F.lit(bounds["min_d"]), F.lit(bounds["max_d"]), F.expr("interval 1 day"))
        ).alias("full_date")
    )
)

print(df_dates.count())  # one row per calendar day, including days with no transactions

In [0]:
dim_date = (
    df_dates
    # Smart key like 20261011: readable and sortable (accepted exception to surrogate keys)
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    # In Spark dayofweek: 1 = Sunday ... 7 = Saturday
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_date")
)

In [0]:
gold_dates = spark.table("workspace.gold.dim_date")
gold_dates.printSchema()
gold_dates.orderBy("full_date").show(5, truncate=False)

# Every transaction date must exist in the calendar. Expected: 0 missing dates
missing = silver.select("transaction_date").distinct().join(
    gold_dates, silver["transaction_date"] == gold_dates["full_date"], "left_anti"
).count()
print("missing dates:", missing)